## Extract channels and merge the data

In [4]:
import os
import glob
import pandas as pd


# =========================
# 1. Path configuration
# =========================
input_dir = r"D:/MyProjects/EEGDatasets/EPOCX/p1/data"
extracted_dir = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_extracted/p1/all_channels"
merged_output = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_merged/p1/all_channels/all_merged.csv"

os.makedirs(extracted_dir, exist_ok=True)
os.makedirs(os.path.dirname(merged_output), exist_ok=True)


# =========================
# 2. Label mapping
#    key: file name without extension, or file name prefix
#    value: class label
# =========================
label_map = {
    "gaming": "concentrate",
    "resting1": "relax",
    "resting2": "relax"
}


# =========================
# 3. Frontal channels and bands
# =========================
frontal_channels = ["AF3", "F7", "F3", "FC5", "FC6", "F4", "F8", "AF4", "T7", "T8", "P8", "O1", "O2"]
bands = ["Theta", "Alpha", "BetaL", "BetaH", "Gamma"]

selected_columns = [
    f"POW.{ch}.{band}"
    for ch in frontal_channels
    for band in bands
]


# =========================
# 4. Get label from file name
#    Return None if no label is found
# =========================
def get_label_from_filename(file_path, label_map):
    base_name = os.path.splitext(os.path.basename(file_path))[0]

    if base_name in label_map:
        return label_map[base_name]

    for prefix, label in label_map.items():
        if base_name.startswith(prefix):
            return label

    return None


# =========================
# 5. Extract frontal band power from one file
# =========================
def extract_frontal_pow(csv_path, output_dir, label_map):
    df = pd.read_csv(csv_path, skiprows=1)

    label_value = get_label_from_filename(csv_path, label_map)
    if label_value is None:
        print(f"Skipped: {os.path.basename(csv_path)} (no label found)")
        return None

    missing_cols = [col for col in selected_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing columns in {os.path.basename(csv_path)}:\n" + "\n".join(missing_cols)
        )

    df_out = df[selected_columns].copy()
    df_out["label"] = label_value

    base_name = os.path.splitext(os.path.basename(csv_path))[0]
    output_path = os.path.join(output_dir, f"{base_name}_frontal.csv")
    df_out.to_csv(output_path, index=False, encoding="utf-8-sig")

    return output_path


# =========================
# 6. Batch extraction
# =========================
all_csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
extracted_files = []

for csv_file in all_csv_files:
    try:
        out_file = extract_frontal_pow(csv_file, extracted_dir, label_map)
        if out_file is not None:
            extracted_files.append(out_file)
            print(f"Processed: {os.path.basename(csv_file)} -> {os.path.basename(out_file)}")
    except Exception as e:
        print(f"Failed: {os.path.basename(csv_file)}")
        print(e)


# =========================
# 7. Merge all extracted files
# =========================
if extracted_files:
    merged_df = pd.concat([pd.read_csv(f) for f in extracted_files], ignore_index=True)
    merged_df.to_csv(merged_output, index=False, encoding="utf-8-sig")
    print(f"\nMerged file saved to: {merged_output}")
    print(f"Merged shape: {merged_df.shape}")
else:
    print("No extracted files to merge.")

Processed: gaming.csv -> gaming_frontal.csv
Processed: resting1.csv -> resting1_frontal.csv
Processed: resting2.csv -> resting2_frontal.csv

Merged file saved to: D:/MyProjects/EEGDatasets/EPOCX/EPOCX_merged/p1/all_channels/all_merged.csv
Merged shape: (3367, 66)


## ANOVA feature selection

In [4]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests

# ========= 1) Load data =========
file_path = r"D:\MyProjects\EEGDatasets\EPOCX\EPOCX_merged\all_channels\all_merged.csv"
df = pd.read_csv(file_path)

# Label column name
LABEL_COL = "label"

# ========= 2) Auto-detect band power feature columns =========
bands = ["delta", "theta", "alpha", "beta", "gamma"]
feature_cols = [
    c for c in df.columns
    if any(b in c.lower() for b in bands) and pd.api.types.is_numeric_dtype(df[c])
]

if LABEL_COL not in df.columns:
    raise ValueError(f"Label column not found: {LABEL_COL}")

if len(feature_cols) == 0:
    raise ValueError(
        "No band power features detected. Please check whether column names contain "
        "delta/theta/alpha/beta/gamma."
    )

# Keep required columns only and remove missing values
data = df[[LABEL_COL] + feature_cols].copy()
data = data.dropna(subset=[LABEL_COL])
for c in feature_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")
data = data.dropna(subset=feature_cols)

# ========= 3) Run one-way ANOVA feature by feature =========
results = []

y = data[LABEL_COL]
classes = sorted(y.unique())

if len(classes) < 2:
    raise ValueError("At least two classes are required to run ANOVA.")

for feat in feature_cols:
    groups = [data.loc[y == cls, feat].values for cls in classes]
    groups = [g for g in groups if len(g) > 1]  # Keep groups with at least 2 samples

    if len(groups) < 2:
        continue

    # ANOVA
    F, p = f_oneway(*groups)

    # Effect size: eta squared
    x = data[feat].values
    grand_mean = np.mean(x)
    ss_between = 0.0
    for cls in classes:
        g = data.loc[y == cls, feat].values
        if len(g) == 0:
            continue
        ss_between += len(g) * (np.mean(g) - grand_mean) ** 2
    ss_total = np.sum((x - grand_mean) ** 2)
    eta2 = ss_between / ss_total if ss_total > 0 else np.nan

    results.append([feat, F, p, eta2])

res = pd.DataFrame(results, columns=["feature", "F", "p", "eta2"])

# ========= 4) Multiple testing correction (FDR-BH) =========
rej, qvals, _, _ = multipletests(res["p"].values, alpha=0.05, method="fdr_bh")
res["q"] = qvals
res["significant_fdr"] = rej

# ========= 5) Rank and select =========
res = res.sort_values(["q", "F"], ascending=[True, False]).reset_index(drop=True)

# Recommended selection rule: q < 0.05 and eta2 >= 0.01
selected = res[(res["q"] < 0.05) & (res["eta2"] >= 0.01)].copy()

print("Total features:", len(res))
print("FDR-significant features:", int(res["significant_fdr"].sum()))
print("Final selected features (q < 0.05 & eta2 >= 0.01):", len(selected))
print("\nTop 20 features:")
print(res.head(20).to_string(index=False))

# Save outputs
# res.to_csv("anova_all_features.csv", index=False)
# selected.to_csv("anova_selected_features.csv", index=False)
# print("\nSaved: anova_all_features.csv, anova_selected_features.csv")


Total features: 65
FDR-significant features: 55
Final selected features (q < 0.05 & eta2 >= 0.01): 47

Top 20 features:
      feature           F             p     eta2             q  significant_fdr
 POW.F4.Alpha 2848.499371  0.000000e+00 0.458585  0.000000e+00             True
 POW.F3.Alpha 2524.456601  0.000000e+00 0.428786  0.000000e+00             True
POW.AF3.Alpha 2409.267771  0.000000e+00 0.417387  0.000000e+00             True
 POW.F8.Alpha 2377.845757  0.000000e+00 0.414198  0.000000e+00             True
POW.AF4.Alpha 2292.179071  0.000000e+00 0.405324  0.000000e+00             True
POW.FC6.Alpha 2140.114266  0.000000e+00 0.388891  0.000000e+00             True
POW.FC5.Alpha 1830.866240 9.083397e-320 0.352505 8.434590e-319             True
 POW.F7.Alpha 1756.467877 3.164307e-309 0.343096 2.570999e-308             True
 POW.T7.Alpha 1619.940674 1.783467e-289 0.325097 1.288060e-288             True
 POW.O2.Alpha  903.143098 5.779300e-176 0.211700 3.756545e-175             True


## train and save the model

In [2]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =========================
# 1. Path configuration
# =========================

output_dir = os.path.dirname(merged_output)
os.makedirs(output_dir, exist_ok=True)

results_path = os.path.join(output_dir, "results_logreg.txt")
save_model = os.path.join(output_dir, "epocx_logreg_model.joblib")
save_scaler = os.path.join(output_dir, "epocx_scaler.joblib")
save_meta = os.path.join(output_dir, "epocx_feature_names.joblib")


# =========================
# 2. Read merged data
# =========================
df = pd.read_csv(merged_output)

print("Data shape:", df.shape)
print(df.head())


# =========================
# 3. Separate features and labels
# =========================
X = df.drop(columns=["label"])
y = df["label"]

feature_names = X.columns.tolist()

print("\nLabel distribution:")
print(y.value_counts())


# =========================
# 4. Split train / test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# 5. Standardization
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================
# 6. Logistic Regression
# =========================
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)


# =========================
# 7. Prediction
# =========================
y_pred = model.predict(X_test)


# =========================
# 8. Evaluation
# =========================
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\nAccuracy:", acc)
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix:")
print(cm)


# =========================
# 9. Save results
# =========================
with open(results_path, "w", encoding="utf-8") as f:
    f.write(f"Merged data path: {merged_output}\n")
    f.write(f"Data shape: {df.shape}\n\n")

    f.write("Label distribution:\n")
    f.write(y.value_counts().to_string())
    f.write("\n\n")

    f.write(f"Accuracy: {acc}\n\n")

    f.write("Classification Report:\n")
    f.write(report)
    f.write("\n")

    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n")

print(f"\nSaved results to {results_path}")


# =========================
# 10. Save model and preprocessing files
# =========================
joblib.dump(model, save_model)
joblib.dump(scaler, save_scaler)
joblib.dump(feature_names, save_meta)

print(f"Saved model to {save_model}")
print(f"Saved scaler to {save_scaler}")
print(f"Saved feature names to {save_meta}")

Data shape: (3365, 66)
   POW.AF3.Theta  POW.AF3.Alpha  POW.AF3.BetaL  POW.AF3.BetaH  POW.AF3.Gamma  \
0       4.456561       3.495647       0.839779       0.325413       0.365118   
1       4.559149       2.995620       0.647689       0.346526       0.353810   
2       4.543970       2.642398       0.543687       0.388384       0.341476   
3       4.526125       2.499658       0.570819       0.450692       0.334778   
4       4.524489       2.525326       0.731554       0.528926       0.334570   

   POW.F7.Theta  POW.F7.Alpha  POW.F7.BetaL  POW.F7.BetaH  POW.F7.Gamma  ...  \
0      3.269726      1.190943      1.039884      0.253717      0.246723  ...   
1      3.316445      1.317984      0.916580      0.253443      0.233739  ...   
2      3.418697      1.674606      0.825198      0.278880      0.236419  ...   
3      3.412848      2.231747      0.802202      0.325729      0.259437  ...   
4      3.224709      2.861127      0.851462      0.385452      0.300356  ...   

   POW.O1.Alpha

## load model and evaluate

In [16]:
import os
import joblib
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =========================
# 1. Path configuration
# =========================
input_dir = r"D:/MyProjects/EEGDatasets/EPOCX/data"
extracted_dir = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_extracted/"
model_dir = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_merged"

model_path = os.path.join(model_dir, "epocx_logreg_model.joblib")
scaler_path = os.path.join(model_dir, "epocx_scaler.joblib")
feature_path = os.path.join(model_dir, "epocx_feature_names.joblib")

# file to evaluate
test_file = os.path.join(extracted_dir, "resting_eo_frontal.csv")


# =========================
# 2. Load model artifacts
# =========================
model = joblib.load(model_path)
scaler = joblib.load(scaler_path)
feature_names = joblib.load(feature_path)


# =========================
# 3. Read new data
# =========================
df_new = pd.read_csv(test_file)

print("Test file:", test_file)
print("Data shape:", df_new.shape)
print(df_new.head())


# =========================
# 4. Prepare features and labels
# =========================
X_new = df_new[feature_names]
y_true = df_new["label"]


# =========================
# 5. Standardize and predict
# =========================
X_new_scaled = scaler.transform(X_new)
y_pred = model.predict(X_new_scaled)

print("\nPredictions:")
print(y_pred[:10])


# =========================
# 6. Evaluate
# =========================
acc = accuracy_score(y_true, y_pred)
all_labels = model.classes_

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
report = classification_report(
    y_true,
    y_pred,
    labels=all_labels,
    target_names=all_labels,
    zero_division=0
)

print("\nAccuracy:", acc)

print("\nClassification Report:")
print(report)

print("\nConfusion Matrix:")
print(cm)

Test file: D:/MyProjects/EEGDatasets/EPOCX/EPOCX_extracted/resting_eo_frontal.csv
Data shape: (961, 41)
   POW.AF3.Theta  POW.AF3.Alpha  POW.AF3.BetaL  POW.AF3.BetaH  POW.AF3.Gamma  \
0       2.105711       3.917364       0.610083       1.355608       0.613861   
1       2.290133       2.560914       0.648077       1.206246       0.626369   
2       2.550823       1.573863       0.666392       1.038230       0.617939   
3       2.680872       0.973034       0.671463       0.861181       0.595879   
4       2.513756       0.685819       0.666736       0.697167       0.573787   

   POW.F7.Theta  POW.F7.Alpha  POW.F7.BetaL  POW.F7.BetaH  POW.F7.Gamma  ...  \
0      1.661901      1.933134      0.762560      0.500692      0.289785  ...   
1      1.888214      1.311046      0.793132      0.381239      0.285571  ...   
2      1.937788      0.853555      0.784725      0.303348      0.268050  ...   
3      1.802785      0.562660      0.758219      0.269292      0.243954  ...   
4      1.517594